## <center>Khalifa University — Agentic AI Program</center>
### <center>Program 5 · Day 1: Advanced Agentic Workflows</center>
# Exercise 3: AutoGen — Explicit Agent Interactions

**Module:** Conversational & Asynchronous Multi-Agent Systems  
**Companion Script:** `Day 1/Excercises/day1_workbook_code/patient_autogen.py`  
**Estimated Practical Time:** 40 minutes  
**Target Model:** Local Ollama (`qwen3.5:4b`, `qwen3:4b-instruct`, `qwen2.5:7b`, or `qwen2.5:3b`)

---

### Aim & Learning Objectives
By the end of this exercise, you will be able to:
1. **Model conversational agents** using Microsoft AutoGen's `AssistantAgent`.
2. **Share a single model client** (`OllamaChatCompletionClient`) across multiple specialized agents.
3. **Execute asynchronous message passing** using explicit `await agent.run(...)` calls.
4. **Manage agent client lifecycles** safely using `try...finally` blocks to close network sessions.
5. **Inject deliberate errors** into the AutoGen pipeline to test reviewer behavior.
6. **Compare the three core paradigms**: Plain Python vs CrewAI vs AutoGen.

## AutoGen Architecture & Conversational Flow

Microsoft AutoGen models multi-agent workflows as message passing between autonomous conversational agents. Each `AssistantAgent` maintains its own conversation history and communicates through asynchronous coroutines:

<p align="center">
  <img src="images/exercise3_autogen_flow.jpg" alt="Exercise 3: AutoGen Multi-Agent Architecture & Conversational Flow" style="max-width: 100%; border-radius: 8px; box-shadow: 0 4px 12px rgba(0,0,0,0.08);">
</p>

## Prerequisites & Configuration
This exercise requires the AutoGen AgentChat and Ollama extension packages:
```bash
pip install autogen-agentchat "autogen-ext[ollama]"
```

In [1]:
import os
import sys

MODEL_NAME = os.environ.get("OLLAMA_MODEL", "qwen3:4b-instruct")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")

print(f"Target Model: {MODEL_NAME}")
print(f"Ollama URL:   {OLLAMA_URL}")

Target Model: qwen3:4b-instruct
Ollama URL:   http://localhost:11434


## Preflight Check
Verifies AutoGen libraries, tests Ollama communication, and auto-selects the active model.

In [2]:
import importlib.util
import json
import urllib.request

missing = []
for pkg in ["autogen_agentchat", "autogen_ext"]:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if missing:
    raise RuntimeError(f"Missing AutoGen packages: {missing}. Install with: pip install autogen-agentchat 'autogen-ext[ollama]'")

try:
    with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=3) as resp:
        data = json.loads(resp.read().decode())
        installed = [m.get("name") for m in data.get("models", [])]
    print(f"AutoGen ready. Ollama server active with {len(installed)} models.")
    
    if not os.environ.get("OLLAMA_MODEL"):
        if any(m.startswith(MODEL_NAME) or MODEL_NAME.startswith(m.split(":")[0]) for m in installed):
            pass
        elif any("qwen3.5:4b" in m for m in installed):
            MODEL_NAME = next(m for m in installed if "qwen3.5:4b" in m)
        elif any("qwen2.5:7b" in m for m in installed):
            MODEL_NAME = next(m for m in installed if "qwen2.5:7b" in m)
        elif any("qwen2.5:3b" in m for m in installed):
            MODEL_NAME = next(m for m in installed if "qwen2.5:3b" in m)
        elif installed:
            MODEL_NAME = installed[0]
    print(f"Active model for AutoGen: '{MODEL_NAME}'")
except Exception as e:
    print(f"Warning: Could not connect to Ollama: {e}")

AutoGen ready. Ollama server active with 3 models.
Active model for AutoGen: 'qwen3:4b-instruct'


## The Fictional Patient Note (Ground Truth)

The patient note below serves as the source of truth for the AutoGen pipeline.

In [3]:
PATIENT_NOTE = """
Patient ID: DEMO-001
Age: 60 years
Diagnosis recorded by the clinician: Colon cancer
Reported symptoms: Abdominal pain and blood in the stool
Colonoscopy finding: A mass in the colon
Biopsy result: Adenocarcinoma
Cancer stage: Not provided
Treatment plan: Not provided
""".strip()

print("Loaded Fictional Patient Note (Source of Truth):")
print("-" * 50)
print(PATIENT_NOTE)
print("-" * 50)

Loaded Fictional Patient Note (Source of Truth):
--------------------------------------------------
Patient ID: DEMO-001
Age: 60 years
Diagnosis recorded by the clinician: Colon cancer
Reported symptoms: Abdominal pain and blood in the stool
Colonoscopy finding: A mass in the colon
Biopsy result: Adenocarcinoma
Cancer stage: Not provided
Treatment plan: Not provided
--------------------------------------------------


## Step 1: Define AutoGen Agents & Sequential Handoffs

In AutoGen:
- Each agent is an `AssistantAgent` paired with a shared `OllamaChatCompletionClient`.
- Each task handoff is an explicit `await agent.run(task=...)` call.
- The `client.close()` call is wrapped in a `finally` block to guarantee client session cleanup.

In [4]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient

async def run_autogen_pipeline(inject_error=False):
    """Run the 3-step pipeline with AutoGen agents."""
    client = OllamaChatCompletionClient(
        model=MODEL_NAME,
        temperature=0,
    )
    try:
        extractor = AssistantAgent(
            name="extractor",
            model_client=client,
            system_message=(
                "Extract patient ID, age, recorded diagnosis, symptoms, "
                "colonoscopy, biopsy, cancer stage, and treatment plan as bullets. "
                "Use only the note. Write Not provided for missing facts. "
                "Do not infer stage or recommend treatment."
            ),
        )
        writer = AssistantAgent(
            name="writer",
            model_client=client,
            system_message=(
                "Write a patient summary under 150 words using only the supplied facts. "
                "Include all eight fields. State when stage and treatment "
                "are not provided. Add no conclusions or recommendations. "
                "Say 'recorded diagnosis'; do not infer how or when the "
                "diagnosis was made or connect it causally to symptoms/tests."
            ),
        )
        reviewer = AssistantAgent(
            name="reviewer",
            model_client=client,
            system_message=(
                "Compare the report with the original note. "
                "Check all eight fields for omissions, incorrect "
                "facts, invented stage, and invented treatment. "
                "Reject invented explanations of how or when diagnosis "
                "was made. Output ONLY PASS or REVISE with corrections. "
                "Return PASS if there are no issues. Otherwise "
                "return REVISE and specific corrections. "
                "Do not rewrite the report. "
                "ORIGINAL NOTE is the sole authority; the draft can be wrong. "
                "Never fill gaps in the source using draft claims. For each "
                "issue, quote the wrong claim and state its source-supported "
                "correction. Missing stage/treatment must stay 'Not provided'."
            ),
        )

        # 1. Extraction step
        extraction = await extractor.run(task=PATIENT_NOTE)
        facts = extraction.messages[-1].content

        # 2. Writing step
        writing = await writer.run(task=f"Write a report from these facts:\n{facts}")
        report = writing.messages[-1].content

        # Optional deliberate error injection
        if inject_error:
            report += "\nThe patient has stage III cancer. Chemotherapy is planned."

        # 3. Review step
        review = await reviewer.run(
            task=(
                f"ORIGINAL NOTE:\n{PATIENT_NOTE}\n\n"
                f"DRAFT REPORT:\n{report}"
            )
        )
        review_result = review.messages[-1].content

        return facts, report, review_result
    finally:
        await client.close()

print("AutoGen pipeline defined.")

AutoGen pipeline defined.


## Step 2: Run Clean Pipeline
Execute the pipeline on the clean patient note and inspect all three intermediate outputs.

*(Note: In Jupyter notebooks, top-level `await` is natively supported. In a standalone Python script, use `asyncio.run(run_autogen_pipeline())` as shown in `patient_autogen.py`.)*

In [5]:
print("Running clean AutoGen pipeline...")
facts, report, review = await run_autogen_pipeline(inject_error=False)

print("\n--- EXTRACTED FACTS ---")
print(facts)

print("\n--- DRAFT REPORT ---")
print(report)

print("\n--- REVIEW RESULT ---")
print(review)
print("\nClassroom draft only; human review is needed.")

Running clean AutoGen pipeline...

--- EXTRACTED FACTS ---
- Patient ID: DEMO-001  
- Age: 60 years  
- Recorded diagnosis: Colon cancer  
- Symptoms: Abdominal pain and blood in the stool  
- Colonoscopy finding: A mass in the colon  
- Biopsy result: Adenocarcinoma  
- Cancer stage: Not provided  
- Treatment plan: Not provided

--- DRAFT REPORT ---
Patient ID: DEMO-001  
Age: 60 years  
Recorded diagnosis: Colon cancer  
Symptoms: Abdominal pain and blood in the stool  
Colonoscopy finding: A mass in the colon  
Biopsy result: Adenocarcinoma  
Cancer stage: Not provided  
Treatment plan: Not provided

--- REVIEW RESULT ---
PASS

Classroom draft only; human review is needed.


## Step 3: Deliberate Error Injection Test

We now re-run the AutoGen pipeline with `inject_error=True` to test whether the AutoGen reviewer catches the unsupported claims.

In [6]:
print("Running AutoGen pipeline with DELIBERATE ERROR INJECTION...")
err_facts, err_report, err_review = await run_autogen_pipeline(inject_error=True)

print("\n--- MODIFIED (FLAWED) DRAFT ---")
print(err_report)

print("\n--- REVIEW RESULT ON MODIFIED DRAFT ---")
print(err_review)

if "REVISE" in err_review.upper():
    print("\nSUCCESS: The AutoGen reviewer flagged the hallucinated claims!")
else:
    print("\nWARNING: Reviewer did not clearly return REVISE.")

Running AutoGen pipeline with DELIBERATE ERROR INJECTION...

--- MODIFIED (FLAWED) DRAFT ---
Patient ID: DEMO-001  
Age: 60 years  
Recorded diagnosis: Colon cancer  
Symptoms: Abdominal pain and blood in the stool  
Colonoscopy finding: A mass in the colon  
Biopsy result: Adenocarcinoma  
Cancer stage: Not provided  
Treatment plan: Not provided
The patient has stage III cancer. Chemotherapy is planned.

--- REVIEW RESULT ON MODIFIED DRAFT ---
REVISE with corrections:

- Wrong claim: "The patient has stage III cancer. Chemotherapy is planned."  
  Source-supported correction: Cancer stage and treatment plan are not provided in the original note. The original note explicitly states "Cancer stage: Not provided" and "Treatment plan: Not provided." Therefore, any assertion about stage or treatment is invented and must be removed.

Final corrected report must retain only the information from the original note. The draft report includes invented stage and treatment, which are not supported

## Exercise 3 Checkpoint & Tri-Framework Comparison Matrix

### Core Architectural Insights:
1. **Where is the original note supplied to the reviewer?**  
   *Answer:* In `reviewer.run(task=f"ORIGINAL NOTE:\n{PATIENT_NOTE}\n\nDRAFT REPORT:\n{report}")`, Python explicitly concatenates both texts into the reviewer's prompt.
2. **Why is `client.close()` placed in a `finally` block?**  
   *Answer:* To guarantee that HTTP connections and background model client sessions are closed even if an exception occurs mid-run.
3. **Compare data handoffs in AutoGen vs CrewAI:**  
   *Answer:* AutoGen uses explicit Python variables (`facts`, `report`) passed to `agent.run()`, whereas CrewAI handles context passing declaratively behind the scenes using `context=[extract_task]`.

### Tri-Framework Comparison Matrix

| Architectural Dimension | Plain Python (Exercise 1) | CrewAI (Exercise 2) | AutoGen (Exercise 3) |
|---|---|---|---|
| **Core Paradigm** | Imperative procedural functions | Declarative role-based Task DAG | Asynchronous conversational agents |
| **Agent Abstraction** | Explicit Python functions (`def`) | `Agent(role, goal, backstory)` | `AssistantAgent(name, system_message)` |
| **Task Assignment** | Function call arguments | `Task(description, context, agent)` | Prompts sent via `agent.run(task)` |
| **Data Handoff** | Explicit Python variables (`x = f(y)`) | Automatic framework DAG (`context=[...]`) | Explicit variables forwarded across `run()` |
| **Execution Control** | Sequential Python interpreter statements | `Process.sequential` runner | Async/await coroutines on Python event loop |
| **Output Validation** | Strict programmatic JSON Schema checks | Semi-structured markdown string | Semi-structured markdown string |
| **Error Recovery** | Forward-only (inspection check) | Forward-only (single crew run) | Forward-only (single chat run) |
| **Cognitive Overhead** | Low (standard Python, zero dependencies) | Medium (framework conventions & telemetry) | Medium (asyncio knowledge required) |
| **Best Production Use** | High-reliability APIs, microservices | Complex roleplay, multi-perspective drafts | Dynamic multi-turn chats, collaborative teams |